# Example 1

In [1]:
import pyomo.environ as pyo # import pyomo package

Every Pyomo model starts with the above. It tells Python to load the Pyomo Modeling Environment.

In [2]:
# Create an instance of a Concrete model
model = pyo.ConcreteModel()

The above defines a variable "model" to hold the model we are about to construct. 

Concrete models are immediately constructed and data must be present at the time components are defined.

In [3]:
# Build the decision variables
model.x = pyo.Var({1,2}) 

In [4]:
# Build the objective function
model.obj = pyo.Objective(expr = 2*model.x[1]+3*model.x[2], sense = pyo.maximize)

In [5]:
# Describe the constraints
model.con1 = pyo.Constraint(expr = model.x[1]-2*model.x[2] <= 4)
model.con2 = pyo.Constraint(expr = 2*model.x[1]+model.x[2] <= 18)
model.con3 = pyo.Constraint(expr = model.x[2] <= 10)
model.con4 = pyo.Constraint(expr = model.x[1] >= 0)
model.con5 = pyo.Constraint(expr = model.x[2] >= 0)

In [6]:
# Print the model
print()
model.pprint()


1 Var Declarations
    x : Size=2, Index={1, 2}
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :  None :  None :  None : False :  True :  Reals
          2 :  None :  None :  None : False :  True :  Reals

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 2*x[1] + 3*x[2]

5 Constraint Declarations
    con1 : Size=1, Index=None, Active=True
        Key  : Lower : Body          : Upper : Active
        None :  -Inf : x[1] - 2*x[2] :   4.0 :   True
    con2 : Size=1, Index=None, Active=True
        Key  : Lower : Body          : Upper : Active
        None :  -Inf : 2*x[1] + x[2] :  18.0 :   True
    con3 : Size=1, Index=None, Active=True
        Key  : Lower : Body : Upper : Active
        None :  -Inf : x[2] :  10.0 :   True
    con4 : Size=1, Index=None, Active=True
        Key  : Lower : Body : Upper : Active
        None :   0.0 : x[1] :  +Inf :   True
    con5

In [7]:
# Solve the model through the GLPK solver
solver = pyo.SolverFactory('glpk')
solver.solve(model)

{'Problem': [{'Name': 'unknown', 'Lower bound': 38.0, 'Upper bound': 38.0, 'Number of objectives': 1, 'Number of constraints': 5, 'Number of variables': 2, 'Number of nonzeros': 7, 'Sense': 'maximize'}], 'Solver': [{'Status': 'ok', 'Termination condition': 'optimal', 'Statistics': {'Branch and bound': {'Number of bounded subproblems': 0, 'Number of created subproblems': 0}}, 'Error rc': 0, 'Time': 0.00946807861328125}], 'Solution': [OrderedDict({'number of solutions': 0, 'number of solutions displayed': 0})]}

In [8]:
# Print the optimzation results
print()
model.display()  # List of all optimization results
print()
print('Optimal value: ', pyo.value(model.obj))  # Print the value of model.obj (i.e., optimal objective value)


Model unknown

  Variables:
    x : Size=2, Index={1, 2}
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :  None :   4.0 :  None : False : False :  Reals
          2 :  None :  10.0 :  None : False : False :  Reals

  Objectives:
    obj : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True :  38.0

  Constraints:
    con1 : Size=1
        Key  : Lower : Body  : Upper
        None :  None : -16.0 :   4.0
    con2 : Size=1
        Key  : Lower : Body : Upper
        None :  None : 18.0 :  18.0
    con3 : Size=1
        Key  : Lower : Body : Upper
        None :  None : 10.0 :  10.0
    con4 : Size=1
        Key  : Lower : Body : Upper
        None :   0.0 :  4.0 :  None
    con5 : Size=1
        Key  : Lower : Body : Upper
        None :   0.0 : 10.0 :  None

Optimal value:  38.0


## Alternative options for the toy example

In [9]:
import pyomo.environ as pyo
model_alt = pyo.ConcreteModel()

In [10]:
# We can build multiple decision variables
# We can build decision variables with speical domains and/or explicit bounds
model_alt.x1 = pyo.Var(within=pyo.NonNegativeReals)
model_alt.x2 = pyo.Var(bounds=(0,10))

In [11]:
# If 'sense' is omitted, the default is minimization.
model_alt.obj = pyo.Objective(expr = -2*model_alt.x1 - 3*model_alt.x2)

In [12]:
# We can build a constraint list
# 'expr' can be a 3-tuple (LB, expr, UB)
model_alt.cons = pyo.ConstraintList()
model_alt.cons.add(model_alt.x1-2*model_alt.x2 <= 4)
model_alt.cons.add((None, 2*model_alt.x1+model_alt.x2, 18))

In [13]:
solver = pyo.SolverFactory('glpk')
solver.solve(model_alt)
print()
model_alt.display()
print()
print('Optimal value: ', -pyo.value(model_alt.obj))


Model unknown

  Variables:
    x1 : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :   4.0 :  None : False : False : NonNegativeReals
    x2 : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :  10.0 :    10 : False : False :  Reals

  Objectives:
    obj : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True : -38.0

  Constraints:
    cons : Size=2
        Key : Lower : Body  : Upper
          1 :  None : -16.0 :   4.0
          2 :  None :  18.0 :  18.0

Optimal value:  38.0
